[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [GeoPandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)

# Dissolve and Explode &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import matplotlib.pyplot as plt

COUNTRIES_URL = ("https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/data/nordic-countries.geojson")
local_copy = Path("../../data/nordic-countries.geojson")
countries = gpd.read_file(local_copy if local_copy.exists() else COUNTRIES_URL)


def mainland_norway():
    """Norway's outline south of 74 degrees north, without Svalbard, in UTM zone 33N."""
    parts = countries[countries["ADMIN"] == "Norway"].to_crs(25833).explode()
    return parts[parts.geometry.bounds["maxy"] < 8_200_000].union_all()


def station_network(n=300, seed=2026):
    """A made-up register of weather stations on mainland Norway, one row each."""
    rng = np.random.default_rng(seed)
    land = mainland_norway()
    x_min, y_min, x_max, y_max = land.bounds
    east, north = [], []
    while len(east) < n:
        x, y = rng.uniform(x_min, x_max), rng.uniform(y_min, y_max)
        if land.contains(shapely.Point(x, y)):
            east.append(x)
            north.append(y)
    lonlat = gpd.GeoSeries(gpd.points_from_xy(east, north), crs=25833).to_crs(4326)
    return pd.DataFrame({
        "station": [f"NS{i:03d}" for i in range(1, n + 1)],
        "latitude": lonlat.y.round(4).to_numpy(), "longitude": lonlat.x.round(4).to_numpy(),
        "elevation_m": np.clip(rng.gamma(1.6, 220, n), 2, 1900).round(0).astype(int),
        "owner": rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2]),
        "installed": rng.integers(1950, 2025, n),
    })


DISTRICT_CENTERS = {"South": (8.0, 58.6), "East": (10.8, 60.5), "West": (6.0, 60.8), "Mid": (11.0, 63.6),
                    "North": (14.5, 67.0), "Far North": (19.0, 69.4), "Northeast": (25.5, 70.0),
                    "Svalbard": (16.0, 78.5)}


def forecast_districts():
    """Eight made-up forecast districts that cover Norway, in UTM zone 33N."""
    norway = countries[countries["ADMIN"] == "Norway"].to_crs(25833).geometry.iloc[0]
    centers = gpd.GeoSeries(gpd.points_from_xy(*zip(*DISTRICT_CENTERS.values())), crs=4326).to_crs(25833)
    cells = shapely.voronoi_polygons(shapely.MultiPoint(list(centers)),
                                     extend_to=shapely.box(*norway.buffer(200_000).bounds)).geoms
    names = [next(name for name, center in zip(DISTRICT_CENTERS, centers) if cell.contains(center))
             for cell in cells]
    districts = gpd.GeoDataFrame({"district": names}, geometry=[cell.intersection(norway) for cell in cells],
                                 crs=25833)
    return districts.set_index("district").loc[list(DISTRICT_CENTERS)].reset_index()


network = station_network()
stations = gpd.GeoDataFrame(
    network, geometry=gpd.points_from_xy(network["longitude"], network["latitude"]), crs=4326
)

districts = forecast_districts()
REGIONS = {"South": "Southern Norway", "East": "Southern Norway", "West": "Southern Norway",
           "Mid": "Southern Norway", "North": "Northern Norway", "Far North": "Northern Norway",
           "Northeast": "Northern Norway", "Svalbard": "Svalbard"}
print("geopandas", gpd.__version__, "|", len(countries), "countries,", len(stations), "made-up stations,",
      len(districts), "made-up districts")

geopandas 1.2.0 | 5 countries, 300 made-up stations, 8 made-up districts


The worked examples' tables, rebuilt here from the made-up stations and districts so that each answer runs on its own.

In [2]:
placed = gpd.sjoin(stations.to_crs(districts.crs), districts, how="left").drop(columns="index_right")
district_counts = placed.groupby("district").size().reindex(districts["district"], fill_value=0)
district_table = districts.assign(
    region=districts["district"].map(REGIONS),
    stations=district_counts.to_numpy(),
    area_km2=(districts.area / 1e6).round(0).astype(int),
)
regions = district_table.dissolve("region", aggfunc={"stations": "sum", "district": "count"})
nordic = countries.to_crs(3035).union_all()

**1.** Dissolve `district_table` into regions with the total area of each region's districts, by summing `area_km2`, and print it beside the area measured from the merged shapes.

In [3]:
by_sum = district_table.dissolve("region", aggfunc={"area_km2": "sum"})
by_sum["measured_km2"] = (by_sum.area / 1e6).round(0).astype(int)
by_sum.drop(columns="geometry")

,area_km2,measured_km2
region,,
Northern Norway,110790,110790
Southern Norway,214039,214039
Svalbard,62068,62068


The two columns agree to the square kilometer, since the districts cover each region without overlapping.

**2.** Print the number of parts in each of the three regions in `regions`, using `explode` and a count by region.

In [4]:
regions.explode().groupby(level=0).size()

region
Northern Norway    17
Southern Norway     7
Svalbard            9
dtype: int64

`explode` keeps each part's region in the index, so a count grouped on the index gives the parts in each region, 17, 7 and 9.

**3.** Print the mean installation year of the stations in each region, computed from `placed`.

In [5]:
placed.groupby(placed["district"].map(REGIONS))["installed"].mean().rename_axis("region").round(1)

region
Northern Norway    1991.8
Southern Norway    1988.3
Name: installed, dtype: float64

The mean comes from the stations, so a district with more stations counts for more. Svalbard has no row, since it has no stations.

**4.** Explode Sweden's outline in UTM zone 33N and print the number of parts and the area of the largest part in square kilometers.

In [6]:
sweden = countries[countries["ADMIN"] == "Sweden"].to_crs(25833).explode(ignore_index=True)
print("parts:", len(sweden))
print("largest:", round(sweden.area.max() / 1e6), "km²")

parts: 6
largest: 441192 km²


The largest part is the mainland, and the rest are islands in the 1:50m outline.

**5.** Make a GeoDataFrame of the parts of `nordic`, the union of the five countries, and print the five largest areas in square kilometers.

In [7]:
pieces = gpd.GeoDataFrame(geometry=list(nordic.geoms), crs=3035)
pieces["area_km2"] = (pieces.area / 1e6).round(0).astype(int)
pieces.nlargest(5, "area_km2")["area_km2"]

24    2192810
18      38836
7       29352
36      14772
39      13516
Name: area_km2, dtype: int64

`union_all` returned a Shapely geometry with no CRS, so the GeoDataFrame is given the CRS the countries were converted to before the union, EPSG:3035, which keeps areas. The largest part is the mainland of Norway, Sweden, Finland and Russia, which touch along their land borders, so the union joins them into one part. The third, 29,352 km², is Denmark's mainland, which touches only Germany, a country the file does not hold.

---

&#8592; **Back to:** [Dissolve and Explode](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/geopandas-deep-dive/11-dissolve-and-explode.ipynb)  &nbsp;&middot;&nbsp;  [GeoPandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/geopandas-deep-dive.html)
